# Why can hedging strategies carry more risk?

#
<!-- Directional market risk: if the hedge behaves differently from what you expected, losses can be larger than planned.

Leverage: many hedging strategies use leverage to amplify returns, and the same leverage amplifies losses.

Liquidity risk: in stressed markets, some hedges may be difficult to trade in time, especially when derivatives or illiquid assets are involved. -->


---
### Factor in Practice Part 3
# Build your own hedge fund: implementing a hedging strategy in Python

##### "I do not know what the future holds, but I know where I will be waiting for it." - Warren Buffett

### 🎬 @Director Harold
### 🏛 CUHK Financial Engineering Undergraduate Program
### 📈 On the path to a U.S. Master's in Financial Engineering (already admitted)
### 🌐 [Follow my Bilibili channel for quant learning content that is easy to follow and actually useful](https://space.bilibili.com/629573485)

🌟🌟🌟 Let's pull back the curtain on quantitative investing. #HaroldQuantChannel 🌟


---


Load data


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime

# Use a representative subset of S&P 500 stocks
sp500_tickers = [
    'AAPL', 'MSFT', 'AMZN', 'GOOGL', 'META', 'TSLA', 'BRK-B', 'JNJ', 'V', 'WMT',
    'JPM', 'PG', 'MA', 'UNH', 'HD', 'DIS', 'BAC', 'NVDA', 'ADBE', 'CRM',
    'NFLX', 'PYPL', 'INTC', 'VZ', 'T', 'MRK', 'PFE', 'ABT', 'TMO', 'NKE',
    'XOM', 'CVX', 'LLY', 'MCD', 'KO', 'PEP', 'ABBV', 'AVGO', 'COST', 'ACN',
    'TXN', 'DHR', 'NEE', 'UPS', 'PM', 'LIN', 'LOW', 'QCOM', 'SPGI', 'IBM',
    'BMY', 'AMGN', 'GS', 'BLK', 'CAT', 'GE', 'MMM', 'AXP', 'SCHW', 'BKNG'
]

start = '2005-01-01'
end = '2022-12-31'

monthly_data = []
for ticker in sp500_tickers:
    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(start=start, end=end, interval='1mo')
        hist = hist.reset_index()
        hist['stock'] = ticker
        hist['date'] = hist['Date'].dt.strftime('%Y-%m')
        hist['monthly_return'] = hist['Close'].pct_change()
        info = stock.info
        hist['market_cap'] = hist['Close'] * info.get('sharesOutstanding', 1e9)
        hist['value_in_thousand'] = hist['market_cap'] / 1000
        hist['next_month_return'] = hist['monthly_return'].shift(-1)
        monthly_data.append(hist[['stock', 'date', 'value_in_thousand', 'monthly_return', 'next_month_return']])
    except:
        pass

df_mkt = pd.concat(monthly_data, axis=0).dropna(subset=['monthly_return']).reset_index(drop=True)

In [ ]:
df_mkt.reset_index(inplace=True, drop=True)
df_mkt

In [ ]:
df_mkt.drop(columns="index")

In [ ]:
df_mkt = df_mkt.drop(columns=["index"], errors='ignore')
df = df_mkt
unique_dates = df['date'].unique()
unique_dates.sort()

In [ ]:
df = df_mkt
unique_dates = df['date'].unique()
unique_dates.sort()

---

Start with a simple hedging idea. Suppose we believe small-cap stocks are more attractive than large-cap stocks. Then we can buy the smallest 1% of stocks in the market and short the largest 1%. That gives us a hedged portfolio.

The beta of this portfolio should be close to zero, meaning its returns should be less dependent on the market. In principle, that lets us hedge away market beta risk and pursue an alpha strategy instead.

That is why this is also called a market-neutral strategy or a hedged strategy.

---


In [ ]:
long_percentile = 1
short_percentile = 99

strategy_returns = []

for date in unique_dates:

    df_data = df[df['date'] == date]

    long_threshold = np.percentile(df_data['value_in_thousand'], long_percentile)
    short_threshold = np.percentile(df_data['value_in_thousand'], short_percentile)

    long_stock = df_data[df_data["value_in_thousand"] <= long_threshold]['stock']
    short_stock = df_data[df_data["value_in_thousand"] >= short_threshold]['stock']

    df_long_stocks = df_data[df_data['stock'].isin(long_stock)]
    df_short_stocks = df_data[df_data['stock'].isin(short_stock)]

    long_return = df_long_stocks['next_month_return'].mean()
    short_return = df_short_stocks['next_month_return'].mean()

    strategy_return = long_return - short_return
    strategy_returns.append(strategy_return)


In [ ]:
strategy_returns

A visual illustration


In [ ]:
datetime

In [ ]:
unique_dates_unchange = unique_dates.copy()
unique_dates = [datetime.strptime(date, "%Y-%m") for date in unique_dates]

In [ ]:
plt.style.use('seaborn-darkgrid')  
plt.figure(figsize=(10, 6))
plt.plot(unique_dates, strategy_returns, label= "strategy_return", color = "red", linewidth = 2)
plt.xlabel("Date")
plt.ylabel("Returns")
plt.legend()
plt.grid(True)

plt.show()




In [ ]:
# strategy_returns al;ksdjfa;lksd
pd.DataFrame(strategy_returns,columns=['returns']).describe()

In [ ]:
cumulated_returns = np.cumsum(strategy_returns)
plt.figure(figsize=(10, 6))
plt.plot(unique_dates, cumulated_returns, label= "strategy_cumulated_return", color = "red", linewidth = 2)
plt.xlabel("Date")
plt.ylabel("Cumulated_Returns")
plt.legend()
plt.grid(True)
plt.show()

This can make money.
What should we do next?
